In [ ]:
# ============================================================
# PAPER DETECTION + SAFE CROPPING PIPELINE (YOLOv8 + GEOMETRY)
# ============================================================
# PURPOSE:
# - Detect ONLY WHITE PAPER containing a drawing
# - Reject incorrect / partial / noisy detections
# - Extract ALL FOUR EDGES of the paper reliably
# - Crop WITHOUT losing drawing information
# - Produce a clean, dataset-style image for ML training
#
# DESIGN PHILOSOPHY:
# - Reject > Risk false psychological inference
# - YOLO proposes region, GEOMETRY verifies it
# - No silent failures
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# -----------------------------
# LOAD YOLOv8 SEGMENTATION MODEL
# -----------------------------
# NOTE:
# This model is NOT trained on paper.
# We therefore treat it as a REGION PROPOSAL model, NOT a final authority.
yolo = YOLO("yolov8n-seg.pt")

# ============================================================
# STEP 1: YOLOv8 PAPER PROPOSAL (SAFE VERSION)
# ============================================================
def detect_paper_yolo(img):
    """
    Uses YOLOv8 segmentation ONLY to propose candidate regions.
    Returns:
        - polygon (Nx2)
        - binary mask (original image space)
    Rejects:
        - No mask
        - Tiny regions
    """

    results = yolo(img, imgsz=640, conf=0.15, verbose=False)
    r = results[0]

    if r.masks is None:
        raise RuntimeError("YOLO failed to detect any segmentation mask")

    polys = r.masks.xy  # polygons already scaled to ORIGINAL image
    if len(polys) == 0:
        raise RuntimeError("YOLO returned empty mask list")

    # Choose largest polygon (assumption: paper is largest object)
    areas = [cv2.contourArea(p.astype(np.int32)) for p in polys]
    idx = int(np.argmax(areas))
    poly = polys[idx].astype(np.int32)

    if cv2.contourArea(poly) < 0.15 * img.shape[0] * img.shape[1]:
        raise RuntimeError("Detected region too small to be a paper")

    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [poly], 255)

    return poly, mask

# ============================================================
# STEP 2: WHITE PAPER VALIDATION (CRITICAL SAFETY STEP)
# ============================================================
def validate_white_paper(img, mask):
    """
    Rejects:
    - Colored paper
    - Wood / table mistaken as paper
    """

    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    masked_s = s[mask == 255]
    masked_v = v[mask == 255]

    # White paper heuristics (strict by design)
    mean_s = np.mean(masked_s)
    mean_v = np.mean(masked_v)

    if mean_s > 30:
        raise RuntimeError("Rejected: paper is not white (high saturation)")

    if mean_v < 180:
        raise RuntimeError("Rejected: region too dark to be white paper")

# ============================================================
# STEP 3: EDGE EXTRACTION (GEOMETRY > ML)
# ============================================================
def extract_paper_edges(img, mask):
    """
    Extracts clean paper boundary using classical vision.
    This is what guarantees 4 sides.
    """

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.bitwise_and(gray, gray, mask=mask)

    edges = cv2.Canny(gray, 50, 150)

    kernel = np.ones((5, 5), np.uint8)
    edges = cv2.dilate(edges, kernel, iterations=2)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise RuntimeError("No edges found for paper")

    cnt = max(contours, key=cv2.contourArea)
    peri = cv2.arcLength(cnt, True)

    approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)

    if len(approx) != 4:
        raise RuntimeError(f"Rejected: expected 4 edges, found {len(approx)}")

    return approx.reshape(4, 2)

# ============================================================
# STEP 4: PERSPECTIVE-SAFE CROPPING
# ============================================================
def warp_and_crop(img, quad):
    """
    Uses perspective transform to extract the FULL paper.
    This prevents half-crops and skewed cuts.
    """

    # Order points: top-left, top-right, bottom-right, bottom-left
    rect = np.zeros((4, 2), dtype="float32")
    s = quad.sum(axis=1)
    diff = np.diff(quad, axis=1)

    rect[0] = quad[np.argmin(s)]
    rect[2] = quad[np.argmax(s)]
    rect[1] = quad[np.argmin(diff)]
    rect[3] = quad[np.argmax(diff)]

    (tl, tr, br, bl) = rect

    wA = np.linalg.norm(br - bl)
    wB = np.linalg.norm(tr - tl)
    hA = np.linalg.norm(tr - br)
    hB = np.linalg.norm(tl - bl)

    W = int(max(wA, wB))
    H = int(max(hA, hB))

    dst = np.array([
        [0, 0],
        [W - 1, 0],
        [W - 1, H - 1],
        [0, H - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(img, M, (W, H))

    return warped

# ============================================================
# STEP 5: FULL SAFE PIPELINE
# ============================================================
def full_pipeline(image_path, visualize=True):
    """
    COMPLETE SAFE PIPELINE
    Any uncertainty → REJECT IMAGE
    """

    img = cv2.imread(image_path)
    if img is None:
        raise RuntimeError("Image not readable")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # YOLO proposal
    poly, mask = detect_paper_yolo(img)

    # White-paper validation
    validate_white_paper(img, mask)

    # Edge refinement
    quad = extract_paper_edges(img, mask)

    # Perspective crop
    cropped = warp_and_crop(img, quad)

    if visualize:
        vis = img.copy()
        cv2.polylines(vis, [quad.astype(int)], True, (255, 0, 0), 4)

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(vis)
        plt.title("Detected Paper Edges")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(mask, cmap="gray")
        plt.title("YOLO Mask")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(cropped)
        plt.title("Final Cropped Paper")
        plt.axis("off")

        plt.show()

    return cropped

# ============================================================
# USAGE
# ============================================================
final_img = full_pipeline("ml-models/image/data/wood_bg1.jpeg")
cv2.imwrite("ml-models/image/data/output/final_safe_output.png", cv2.cvtColor(final_img, cv2.COLOR_RGB2BGR))